# 06 Kernel PCA + 线性回归 多因子选股

## 论文参考
- **作者**: Mei-li Zhou
- **年份**: 2020
- **标题**: *Quantitative Stock Selection Strategies Based on Kernel Principal Component Analysis*
- **DOI**: 10.12783/dtetr/mcaee2020/35038
- **摘要**: 本文提出基于核主成分分析(KPCA)的量化选股策略。传统PCA仅能捕捉因子间的线性关系，而KPCA通过RBF核函数将因子映射到高维特征空间，提取非线性主成分。在降维后的特征空间中使用线性回归预测收益，实证显示策略年化收益率达41%，夏普比率约3。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### 标准PCA
对中心化数据矩阵 $X \in \mathbb{R}^{n \times p}$，求协方差矩阵特征分解:
$$C = \frac{1}{n} X^T X, \quad C v_k = \lambda_k v_k$$
投影: $Z = X V_d$ ($V_d$ 取前 $d$ 个特征向量)

### Kernel PCA (KPCA)
通过核技巧，在高维隐式特征空间 $\mathcal{F}$ 中执行PCA:

1. **核映射**: $\phi: \mathbb{R}^p \to \mathcal{F}$，使用RBF核函数:
$$K(x_i, x_j) = \exp\left(-\frac{\| x_i - x_j \|^2}{2\sigma^2}\right) = \langle \phi(x_i), \phi(x_j) \rangle$$

2. **核矩阵特征分解**: $K \alpha_k = n \lambda_k \alpha_k$

3. **中心化核矩阵**: $\tilde{K} = K - \mathbf{1}_n K - K \mathbf{1}_n + \mathbf{1}_n K \mathbf{1}_n$

4. **投影**: 新样本 $x$ 在第 $k$ 个主成分上的投影:
$$z_k = \sum_{i=1}^n \alpha_k^{(i)} K(x_i, x)$$

### 选股策略
1. 每月末对15个因子执行KPCA降维 -> 提取前5个核主成分
2. 用核主成分作为特征，线性回归预测下月收益
3. 选预测收益最高的Top 10持有
4. 同时用标准PCA对比

In [ ]:
# === Cell 3: KPCA 3D可视化动画 ===
import numpy as np
import plotly.graph_objects as go
from sklearn.decomposition import KernelPCA, PCA
from sklearn.datasets import make_circles

# 生成非线性可分数据 (同心圆)
np.random.seed(42)
X_circle, y_circle = make_circles(n_samples=400, factor=0.3, noise=0.1)

# 标准PCA (2D -> 2D)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_circle)

# KPCA (2D -> 3D for visualization)
kpca = KernelPCA(n_components=3, kernel='rbf', gamma=5)
X_kpca = kpca.fit_transform(X_circle)

# Plotly 3D scatter for KPCA result
fig = go.Figure()

# KPCA 3D scatter
for label, color, name in [(0, '#2196F3', '类别0 (外环)'), (1, '#F44336', '类别1 (内环)')]:
    mask = y_circle == label
    fig.add_trace(go.Scatter3d(
        x=X_kpca[mask, 0], y=X_kpca[mask, 1], z=X_kpca[mask, 2],
        mode='markers', name=name,
        marker=dict(size=3, color=color, opacity=0.7),
    ))

# Add rotation animation frames
frames = []
for angle in range(0, 360, 10):
    frames.append(go.Frame(
        layout=dict(
            scene=dict(
                camera=dict(
                    eye=dict(
                        x=1.5 * np.cos(np.radians(angle)),
                        y=1.5 * np.sin(np.radians(angle)),
                        z=0.8
                    )
                )
            )
        ),
        name=f'{angle}deg'
    ))

fig.frames = frames
fig.update_layout(
    title=dict(text="Kernel PCA (RBF): 非线性数据在核特征空间中线性可分"),
    scene=dict(
        xaxis_title="KPC 1", yaxis_title="KPC 2", zaxis_title="KPC 3",
    ),
    height=600, width=900,
    updatemenus=[dict(
        type="buttons",
        buttons=[
            dict(label="\u25b6 旋转", method="animate",
                 args=[None, {"frame": {"duration": 100}, "transition": {"duration": 50}}]),
            dict(label="\u23f8 暂停", method="animate",
                 args=[[None], {"frame": {"duration": 0}, "mode": "immediate"}]),
        ]
    )],
)
fig.show()

In [ ]:
# === Cell 4: 导入与配置 ===
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.decomposition import KernelPCA, PCA
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from shared.data_fetcher import get_stock_daily, get_index_daily, get_multiple_stocks_daily
from shared.factors import build_factor_panel, winsorize, standardize
from shared.backtest_engine import MultiStockBacktester
from shared.visualization import (plot_equity_curve, plot_drawdown,
                                   plot_metrics_table, plot_cumulative_comparison)
from shared.constants import *

print("All imports successful.")

In [ ]:
# === Cell 5: 数据获取 ===
STOCK_POOL = [
    "600519", "601318", "600036", "000858", "601166",
    "600276", "601398", "600030", "000333", "002415",
    "600900", "601888", "600809", "000568", "002304",
    "601012", "600031", "603259", "600585", "000661",
]

stock_data = get_multiple_stocks_daily(STOCK_POOL, DEFAULT_START, DEFAULT_END)
benchmark = get_index_daily(CSI300_CODE, DEFAULT_START, DEFAULT_END)

print(f"Successfully loaded {len(stock_data)} stocks")
print(f"Benchmark: {len(benchmark)} trading days")

In [ ]:
# === Cell 6: 因子工程 ===
panel = build_factor_panel(stock_data)
print(f"Factor panel shape: {panel.shape}")

FEATURE_COLS = [
    'mom_5', 'mom_10', 'mom_20', 'mom_60',
    'vol_5', 'vol_10', 'vol_20',
    'price_to_ma_5', 'price_to_ma_10', 'price_to_ma_20',
    'rsi', 'macd_hist', 'bb_pctb', 'bb_width', 'vp_corr',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in panel.columns]
TARGET_COL = 'fwd_return_20d'

for col in FEATURE_COLS:
    panel[col] = panel.groupby('date')[col].transform(
        lambda x: standardize(winsorize(x))
    )

panel.dropna(subset=FEATURE_COLS + [TARGET_COL], inplace=True)
print(f"After cleaning: {panel.shape}")
print(f"Features: {len(FEATURE_COLS)}")

In [ ]:
# === Cell 7: 滚动训练 - KPCA vs PCA ===
panel['date'] = pd.to_datetime(panel['date'])
panel['year_month'] = panel['date'].dt.to_period('M')
months = sorted(panel['year_month'].unique())

TRAIN_WINDOW = 6
TOP_N = 10
N_COMPONENTS = 5

model_configs = {
    'PCA + LR': Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=N_COMPONENTS)),
        ('lr', LinearRegression()),
    ]),
    'KPCA(RBF) + LR': Pipeline([
        ('scaler', StandardScaler()),
        ('kpca', KernelPCA(n_components=N_COMPONENTS, kernel='rbf', gamma=0.1)),
        ('lr', LinearRegression()),
    ]),
    'KPCA(poly) + LR': Pipeline([
        ('scaler', StandardScaler()),
        ('kpca', KernelPCA(n_components=N_COMPONENTS, kernel='poly', degree=3)),
        ('lr', LinearRegression()),
    ]),
}

selections = {name: {} for name in model_configs}
explained_variance_history = []

for i in range(TRAIN_WINDOW, len(months) - 1):
    train_months = months[i - TRAIN_WINDOW:i]
    train_mask = panel['year_month'].isin(train_months)
    train_df = panel[train_mask]

    pred_month = months[i]
    pred_mask = panel['year_month'] == pred_month
    pred_df = panel[pred_mask]

    if len(train_df) < 50 or len(pred_df) < 5:
        continue

    X_train = train_df[FEATURE_COLS].values
    y_train = train_df[TARGET_COL].values
    X_pred = pred_df[FEATURE_COLS].values

    for name, pipe in model_configs.items():
        try:
            # Clone pipeline to avoid state issues
            from sklearn.base import clone
            model = clone(pipe)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_pred)

            # Track PCA explained variance
            if 'pca' in model.named_steps and hasattr(model.named_steps['pca'], 'explained_variance_ratio_'):
                explained_variance_history.append(model.named_steps['pca'].explained_variance_ratio_)

            pred_df_copy = pred_df.copy()
            pred_df_copy['predicted'] = y_pred
            top_stocks = pred_df_copy.nlargest(TOP_N, 'predicted')['symbol'].tolist()

            rebal_date = pred_df_copy['date'].max()
            selections[name][rebal_date] = top_stocks

        except Exception as e:
            continue

for name in selections:
    print(f"{name}: {len(selections[name])} rebalance periods")

In [ ]:
# === Cell 8: 回测 ===
all_prices = {sym: stock_data[sym]['close'] for sym in stock_data}
bench_close = benchmark['close'] if 'close' in benchmark.columns else None

results = {}
for name in model_configs:
    bt = MultiStockBacktester(
        initial_capital=INITIAL_CAPITAL,
        rebalance_freq='M'
    )
    result = bt.run(all_prices, selections[name], bench_close)
    results[name] = result
    print(f"\n=== {name} ===")
    for k, v in result['metrics'].items():
        print(f"  {k}: {v}")

In [ ]:
# === Cell 9: 可视化 ===
import matplotlib.pyplot as plt

# 1. 三模型累计收益对比
strategy_returns = {name: results[name]['returns'] for name in results}
plot_cumulative_comparison(strategy_returns, title="PCA vs KPCA(RBF) vs KPCA(poly) 累计收益对比")

# 2. 最佳模型收益曲线 vs 基准
best_model = max(results, key=lambda k: float(results[k]['metrics']['年化收益率'].strip('%')))
bench_eq = None
if results[best_model].get('benchmark_returns') is not None:
    bench_eq = INITIAL_CAPITAL * (1 + results[best_model]['benchmark_returns']).cumprod()
plot_equity_curve(results[best_model]['equity_curve'], bench_eq,
                  title=f"最优模型 ({best_model}) vs 沪深300")

# 3. 回撤
plot_drawdown(results[best_model]['equity_curve'],
              title=f"最优模型 ({best_model}) - 回撤")

# 4. PCA解释方差比例
if explained_variance_history:
    avg_ev = np.mean(explained_variance_history, axis=0)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(range(1, len(avg_ev)+1), avg_ev, color='#2196F3', alpha=0.8, label='各主成分')
    ax.plot(range(1, len(avg_ev)+1), np.cumsum(avg_ev), 'ro-', label='累计解释比例')
    ax.set_xlabel('主成分序号', fontsize=12)
    ax.set_ylabel('解释方差比例', fontsize=12)
    ax.set_title('PCA 解释方差比例 (KPCA无此指标)', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# 5. 绩效对比表
fig, ax = plt.subplots(figsize=(12, 5))
ax.axis('off')
metrics_keys = ['累计收益率', '年化收益率', '夏普比率', '最大回撤', '胜率', 'Calmar比率']
cell_text = []
for key in metrics_keys:
    row = [key] + [results[m]['metrics'].get(key, 'N/A') for m in model_configs]
    cell_text.append(row)
table = ax.table(cellText=cell_text,
                 colLabels=['指标'] + list(model_configs.keys()),
                 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)
for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_facecolor('#2196F3')
        cell.set_text_props(color='white', fontweight='bold')
    elif i % 2 == 0:
        cell.set_facecolor('#f0f0f0')
ax.set_title("PCA vs KPCA 绩效对比", fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 结果讨论

### KPCA vs PCA 核心差异
1. **PCA**: 线性降维，提取因子间的线性相关结构；计算高效，结果可解释（有明确的解释方差比例）
2. **KPCA (RBF)**: 通过径向基核函数捕捉非线性关系；gamma参数控制核宽度，需要调参
3. **KPCA (Poly)**: 多项式核捕捉多阶交互效应；degree控制多项式阶数

### 降维的价值
- 将15个因子压缩为5个主成分，降低噪声和共线性
- 核方法在因子关系非线性时优势更明显
- 降维后的线性回归复杂度低，不易过拟合

### 与论文对比
- Zhou (2020) 报告KPCA策略年化41%、夏普比率3
- KPCA的gamma参数选择对结果影响较大
- 非线性降维的收益与市场复杂度相关: 因子关系越非线性，KPCA优势越大

### 注意事项
- KPCA的核主成分无法像PCA一样解释为原始因子的线性组合
- 计算复杂度为 $O(n^3)$，大规模数据需考虑Nystrom近似